# 18.2 怎麼讓學生本身更小？


把模型取名student，再讓它向教師學習，權重檔就會自動縮小嗎？不會。蒸餾改變學習目標，不會直接刪掉模型的參數。要得到較小學生，必須先選較少層、較窄特徵，或不同結構，再看教師訊號能否幫它彌補容量減少。前置是[教師提供的訊號](https://birdhackor.github.io/tiny-perceptron-vlm/18.1.html)與[層深度](https://birdhackor.github.io/tiny-perceptron-vlm/4.8.html)；「小」要有實際參數與成本記錄，不是模型名稱。

用相同詞表264項，教師寬度16、兩層，學生寬度8、一層。width是每個位置的特徵數，layers是重複處理特徵與注意力的層數。詞表相同讓兩者的每個輸出候選能對上，內部特徵卻可以不同。較窄學生的查字表、矩陣與中間特徵都會縮小；較少層則減少重複處理的次數。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
from tiny_perceptron.model import TinyLM, ModelConfig

configs = [("教師", ModelConfig(width=16, layers=2)), ("學生", ModelConfig(width=8, layers=1))]
for name, config in configs:
    model = TinyLM(config)
    parameters = sum(p.numel() for p in model.parameters())
    print(name, "寬度", config.width, "層數", config.layers, "參數", parameters, "FP32數字bytes", parameters * 4)

`ModelConfig`描述架構，`TinyLM`按它建立模型；`parameters()`列出可學權重，`numel`逐一數元素。兩個模型仍然是隨機初始化，沒有教師能力，也沒有訓練學生；這段只確認所選的學生真的較小。FP32是32位元浮點數的儲存格式，每個數占32位元；一byte是8位元，所以每個FP32數占32/8=4bytes，程式用參數數量乘4。輸出應為教師16960個參數、67840bytes，學生6104個、24416bytes。這只量權重數字部分，沒有量檔案容器與執行峰值。

寬度減半、層數減半不代表完整參數減到八分之一。token是文字切分後的編碼單位，每項有自己的編號，切分例子見[6.1](https://birdhackor.github.io/tiny-perceptron-vlm/6.1.html)。FFN與注意力的某些矩陣兩個邊都隨寬度變，數量大致平方變化；字嵌入與輸出詞表每個token仍要一列特徵，它們只隨寬度線性變化，也不隨層數重複。比較完整模型時應數所有模組，不能只看一層中的最大矩陣。

正式屬性教師寬64、兩層，共141,568個參數，FP32數值566,272 bytes；兩個學生都一層、兩個attention heads，寬16有13,744個參數／54,976 bytes，寬32有33,632個／134,528 bytes。這些縮小先由架構決定。同寬度的普通CE、教師生成硬目標與CE+KL三支，參數個數和初始化完全相同；其輸出投影也相同，沒有讓蒸餾支線偷偷多一個接頭。但寬16與寬32的投影和表示容量本來就不同，不能跨寬度把所有品質差都歸給教師訊號。

學生更小可能損失表達能力。應先用相同學生架構建立未蒸餾基準，再加入教師文字或分布，才能知道好處來自教師訊號，還是原本學生設置。固定同一學生架構時也需記錄訓練資料、已讀token數、初始化與評估方法；已讀token數是訓練累計處理多少個編碼單位，不是候選詞表有幾項。後面[蒸餾是否有幫助](https://birdhackor.github.io/tiny-perceptron-vlm/18.10.html)會比較這些條件。

練習只把學生width由8改成16、layers仍為1。先預測它比原學生大、比兩層教師小，但不是教師參數恰好一半；再重跑，應得到13744個參數。這時雙方寬度相同、只有層數不同，能看見字嵌入等共享部分為何不隨層數減半。
